# Band Structure

Calculate the electronic band structure of a material using a DFT workflow on the Mat3ra platform.

<h2 style="color:green">Usage</h2>

1. Set material, type of calculation and its parameters in cell 1.2. below (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for material, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder — place files there manually or run a material creation notebook first. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Create workflow and set its parameters: select application, load band structure workflow from Standata, optionally add relaxation or adjust model/method parameters, and save the workflow to the platform.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from material, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the band structure.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters and configurations for the workflow and job


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # Name of the material to load from local file or Standata

# 4. Workflow parameters
# Options: "band_structure" / "band_structure_dos" / "band_structure_hse" /
#          "band_structure_magn" / "band_structure_soc" / "dos"
CALCULATION_TYPE = "band_structure"

APPLICATION_NAME = "espresso"  # Specify application name (e.g., "espresso", "vasp")
ADD_RELAXATION = None  # Whether to add relaxation subworkflow as first unit

WORKFLOW_SEARCH_TERM = f"{CALCULATION_TYPE}.json"
MY_WORKFLOW_NAME = CALCULATION_TYPE.replace("_", " ").title() + (" (relax)" if ADD_RELAXATION else "")

# K-grid for SCF and relaxation steps (if not set, KPPRA is used by default)
RELAXATION_KGRID = None  # e.g., [4, 4, 4]
SCF_KGRID = None  # e.g., [4, 4, 4]

# K-grid for NSCF step — used by "band_structure_dos" and "dos"
NSCF_KGRID = None  # e.g., [8, 8, 8]

# K-path for band structure (if not set, workflow default is used)
# Example for FCC: [{"point": "L", "steps": 20}, {"point": "Г", "steps": 20}, {"point": "X", "steps": 20}]
# Not used by "dos" or "band_structure_hse" (QE EXX limitation: see cell 4.3.3 for details)
KPATH = None

ECUTWFC = 40   # plane-wave cutoff energies in Ry (if None, defaults are used)
ECUTRHO = 200  # density cutoff, typically 4-8x ECUTWFC depending on pseudopotentials (if None, defaults are used)

# Initial magnetic moment per atomic species — used by "band_structure_magn"
# e.g., {"Fe": 4.0, "O": 0.0}
STARTING_MAGNETIZATION = None

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from utils.auth import authenticate

await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.visualize import visualize_materials as visualize
from utils.jupyterlite import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from utils.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Create workflow and set its parameters
### 4.1. Get list of applications and select one

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Create workflow from standard workflows and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Modify workflow (Optional)
#### 4.3.1. Add relaxation

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()


#### 4.3.2. Modify model and method parameters (Optional)
Uncomment the code below and adjust selection of model parameters as needed.

In [ ]:
# Most variants use the model/method embedded in their standata definition:
#   - "band_structure_hse"  → hybrid DFT (HSE06), ultrasoft pseudopotentials
#   - "band_structure_soc"  → norm-conserving fully-relativistic pseudopotentials (nc-fr)
#   - all others            → GGA/LDA with default pseudopotentials
#
# Uncomment and adjust below to override the model for any variant:
#
# from mat3ra.mode.model import Model
# from mat3ra.standata.model_tree import ModelTreeStandata
#
# model_config = ModelTreeStandata.get_model_by_parameters(
#     type="dft",
#     subtype="gga",   # e.g., "lda", "gga", "hybrid"
#     functional="pbe" # e.g., "pz" (LDA), "pbe" (GGA), "hse06" (HSE)
# )
# model_config["method"] = {"type": "pseudopotential", "subtype": "us"}  # e.g., "us", "nc", "nc-fr"
# model = Model.create(model_config)
#
# for subworkflow in workflow.subworkflows:
#     subworkflow.model = model
#
# visualize_workflow(workflow)

#### 4.3.3. Modify important settings
Set k-grid, k-path, and energy cutoffs.

In [ ]:
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider

# For HSE, subworkflows are: [pw_scf, espresso_extract_kpoints, band_structure_hse]
# bs_subworkflow is used for NSCF and STARTING_MAGNETIZATION lookups on the first BS subworkflow
bs_subworkflow = workflow.subworkflows[1 if ADD_RELAXATION else 0]

if RELAXATION_KGRID is not None and ADD_RELAXATION:
    unit = workflow.subworkflows[0].get_unit_by_name(name_regex="relax")
    unit.add_context(PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).yield_data())
    workflow.subworkflows[0].set_unit(unit)

# SCF k-grid for standard variants; also sets qgrid (nqx1/2/3) for HSE — the template reads
# qgrid.dimensions, not kgrid, so both contexts are needed on pw_scf_bands_hse.
if SCF_KGRID is not None:
    for unit_name in ["pw_scf", "pw_scf_magn", "pw_scf_soc"]:
        for swf in workflow.subworkflows:
            unit = swf.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
                swf.set_unit(unit)
    for swf in workflow.subworkflows:
        unit = swf.get_unit_by_name(name="pw_scf_bands_hse")
        if unit:
            unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
            unit.add_context(PointsGridDataProvider(name="qgrid", dimensions=SCF_KGRID, isEdited=True).yield_data())
            swf.set_unit(unit)

# KPATH: applied to pw_bands variants; NOT applied to pw_scf_bands_hse (QE EXX limitation).
if KPATH is not None:
    for unit_name in ["pw_bands", "pw_bands_magn", "pw_bands_soc"]:
        for swf in workflow.subworkflows:
            unit = swf.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).yield_data())
                swf.set_unit(unit)

# NSCF_KGRID: used by "band_structure_dos" and "dos"
if NSCF_KGRID is not None:
    unit = bs_subworkflow.get_unit_by_name(name="pw_nscf")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=NSCF_KGRID, isEdited=True).yield_data())
        bs_subworkflow.set_unit(unit)

if ECUTWFC is not None:
    cutoffs_context = PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).yield_data()
    for unit_name in ["pw_vc-relax", "pw_scf", "pw_scf_magn", "pw_scf_soc", "pw_scf_bands_hse", "pw_bands",
                      "pw_bands_magn", "pw_bands_soc"]:
        for swf in workflow.subworkflows:
            unit = swf.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(cutoffs_context)
                swf.set_unit(unit)

# STARTING_MAGNETIZATION: used by "band_structure_magn"
if STARTING_MAGNETIZATION is not None:
    unit = bs_subworkflow.get_unit_by_name(name="pw_scf_magn")
    if unit:
        magn_lines = "\n".join(
            f"    starting_magnetization({i + 1}) = {val}"
            for i, val in enumerate(STARTING_MAGNETIZATION.values())
        )
        existing = unit.input[0]["content"] if unit.input else ""
        unit.input = [{"name": unit.input[0]["name"] if unit.input else "pw_scf_magn.in",
                       "content": existing.replace("/", f"{magn_lines}\n/")}]
        bs_subworkflow.set_unit(unit)

visualize_workflow(workflow)

### 4.4. Save workflow to collection

In [ ]:
from utils.generic import dict_to_namespace
from utils.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the job

In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with material and workflow configuration
### 6.1. Create job

In [ ]:
from utils.api import create_job
from utils.visualize import display_JSON

print(f"Material: {saved_material.id}")
print(f"Workflow: {saved_workflow.id}")
print(f"Project: {project_id}")

job_name = MY_WORKFLOW_NAME + " " + saved_material.formula + " " + timestamp
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict()
)

job = dict_to_namespace(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve results

In [ ]:
from utils.visualize import visualize_properties

property_data = client.properties.get_for_job(job_id)
visualize_properties(property_data, title="Band Structure")
